# Final Comparison — Centralized vs Federated IDS

This notebook compares the four completed experiments:

1. **Centralized**
2. **FedAvg — IID**
3. **FedAvg — Non-IID**
4. **FedProx — Non-IID**

It performs **no model training**. It only reads the saved result artifacts, verifies that the comparison is fair, builds report-ready tables/plots, and saves one compact comparison package.

## Kaggle inputs

Add the saved notebook outputs of:

- `centralised-baseline-markp`
- `fedavg-iid-optimized-v2`
- `fedavg-non-iid-optimized-v2`
- `fedprox-non-iid-optimized-v2`

You do **not** need the raw CICIoT2023 dataset or the dataset-preparation notebook here.

**GPU is not required.**


In [ ]:
# Cell 02: Imports and output directory
from pathlib import Path
import json
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/final_comparison_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input root:", INPUT_ROOT)
print("Output directory:", OUTPUT_DIR)


In [ ]:
# Cell 03: Locate required experiment artifacts
# We use exact filenames so the notebook cannot silently mix experiments.

REQUIRED_FILES = {
    "central_metrics": "centralized_baseline_optimized_v2_frac_1p000_comparison_metrics.csv",
    "central_results": "centralized_baseline_optimized_v2_frac_1p000_results.json",
    "central_per_class": "centralized_baseline_optimized_v2_frac_1p000_per_class.csv",
    "central_history": "centralized_baseline_optimized_v2_frac_1p000_history.csv",

    "iid_metrics": "fedavg_iid_comparison_metrics.csv",
    "iid_results": "fedavg_iid_results.json",
    "iid_per_class": "fedavg_iid_per_class.csv",
    "iid_rounds": "fedavg_iid_round_history.csv",

    "non_iid_metrics": "fedavg_non_iid_comparison_metrics.csv",
    "non_iid_results": "fedavg_non_iid_results.json",
    "non_iid_per_class": "fedavg_non_iid_per_class.csv",
    "non_iid_rounds": "fedavg_non_iid_round_history.csv",
    "non_iid_positions": "fedavg_non_iid_client_positions.npz",

    "fedprox_metrics": "fedprox_non_iid_comparison_metrics.csv",
    "fedprox_results": "fedprox_non_iid_results.json",
    "fedprox_per_class": "fedprox_non_iid_per_class.csv",
    "fedprox_rounds": "fedprox_non_iid_round_history.csv",
    "fedprox_positions": "fedprox_non_iid_client_positions.npz",
}

def find_exact_file(filename):
    matches = sorted(INPUT_ROOT.rglob(filename))
    if not matches:
        raise FileNotFoundError(
            f"Missing: {filename}\n"
            "Add the saved output of the corresponding experiment notebook in Kaggle."
        )
    if len(matches) > 1:
        print(f"Warning: found {len(matches)} copies of {filename}; using:")
    print(" ", matches[0])
    return matches[0]

paths = {key: find_exact_file(name) for key, name in REQUIRED_FILES.items()}

print("\nAll required artifacts found.")


In [ ]:
# Cell 04: Load the four headline metric rows

def read_single_row_csv(path):
    df = pd.read_csv(path)
    if len(df) != 1:
        raise ValueError(f"Expected exactly one row in {path.name}, found {len(df)}")
    return df.iloc[0].to_dict()

central = read_single_row_csv(paths["central_metrics"])
iid = read_single_row_csv(paths["iid_metrics"])
non_iid = read_single_row_csv(paths["non_iid_metrics"])
fedprox = read_single_row_csv(paths["fedprox_metrics"])

central["experiment"] = "Centralized"

# Normalize experiment labels so later tables have a fixed order.
iid["experiment"] = "FedAvg-IID"
non_iid["experiment"] = "FedAvg-Non-IID"
fedprox["experiment"] = "FedProx-Non-IID"

comparison_df = pd.DataFrame([central, iid, non_iid, fedprox])

EXPERIMENT_ORDER = [
    "Centralized",
    "FedAvg-IID",
    "FedAvg-Non-IID",
    "FedProx-Non-IID",
]
comparison_df["experiment"] = pd.Categorical(
    comparison_df["experiment"],
    categories=EXPERIMENT_ORDER,
    ordered=True,
)
comparison_df = comparison_df.sort_values("experiment").reset_index(drop=True)

print("Loaded headline results:")
display(comparison_df)


In [ ]:
# Cell 05: Load experiment metadata for fairness checks

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

central_json = load_json(paths["central_results"])
iid_json = load_json(paths["iid_results"])
non_iid_json = load_json(paths["non_iid_results"])
fedprox_json = load_json(paths["fedprox_results"])

def central_config(d):
    return {
        "experiment": "Centralized",
        "dataset_frac": d["dataset_frac_requested"],
        "train_samples": d["dataset"]["train_samples"],
        "test_samples": d["dataset"]["test_samples"],
        "features": d["dataset"]["num_features"],
        "classes": d["dataset"]["num_classes"],
        "architecture": d["model"]["architecture"],
        "seed": d["seed"],
        "batch_size": d["training"]["batch_size"],
        "learning_rate": d["training"]["learning_rate"],
        "effective_data_passes": d["training"]["epochs"],
        "partition": "Centralized",
    }

def fl_config(d, label):
    prepared = d["prepared_artifact"]
    training = d["training"]
    partition = d.get("partition", "IID")

    if isinstance(partition, dict):
        partition_name = partition.get("type", "Non-IID")
    else:
        partition_name = str(partition)

    return {
        "experiment": label,
        "dataset_frac": prepared["dataset_frac_requested"],
        "train_samples": prepared["train_samples"],
        "test_samples": prepared["test_samples"],
        "features": prepared["num_features"],
        "classes": prepared["num_classes"],
        "architecture": d["model"]["architecture"],
        "seed": prepared.get("seed", 42),
        "batch_size": training["batch_size"],
        "learning_rate": training["learning_rate"],
        "effective_data_passes": training["effective_data_passes"],
        "partition": partition_name,
    }

config_df = pd.DataFrame([
    central_config(central_json),
    fl_config(iid_json, "FedAvg-IID"),
    fl_config(non_iid_json, "FedAvg-Non-IID"),
    fl_config(fedprox_json, "FedProx-Non-IID"),
])

display(config_df)


In [ ]:
# Cell 06: Validate that the comparison is controlled

CHECK_COLUMNS = [
    "dataset_frac",
    "train_samples",
    "test_samples",
    "features",
    "classes",
    "architecture",
    "seed",
    "batch_size",
    "learning_rate",
    "effective_data_passes",
]

fairness_rows = []
for column in CHECK_COLUMNS:
    values = config_df[column].tolist()
    fairness_rows.append({
        "check": column,
        "consistent": len({str(v) for v in values}) == 1,
        "values": " | ".join(str(v) for v in values),
    })

fairness_df = pd.DataFrame(fairness_rows)

# Stronger check: FedAvg-Non-IID and FedProx-Non-IID should use the exact same
# deterministic client positions.
fedavg_partition = np.load(paths["non_iid_positions"])
fedprox_partition = np.load(paths["fedprox_positions"])

same_partition_keys = set(fedavg_partition.files) == set(fedprox_partition.files)
same_partition_arrays = same_partition_keys and all(
    np.array_equal(fedavg_partition[key], fedprox_partition[key])
    for key in fedavg_partition.files
)

fairness_df = pd.concat([
    fairness_df,
    pd.DataFrame([{
        "check": "FedAvg-Non-IID vs FedProx exact client partition",
        "consistent": bool(same_partition_arrays),
        "values": "Exact match" if same_partition_arrays else "Mismatch",
    }])
], ignore_index=True)

display(fairness_df)

failed_checks = fairness_df.loc[~fairness_df["consistent"], "check"].tolist()
if failed_checks:
    print("\nWARNING — comparison-control checks failed:")
    for item in failed_checks:
        print(" -", item)
else:
    print("\nAll controlled-comparison checks passed.")


In [ ]:
# Cell 07: Report-ready headline comparison table

METRICS = [
    "accuracy",
    "balanced_accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "weighted_f1",
    "mcc",
    "cohen_kappa",
]

headline_df = comparison_df[["experiment"] + METRICS].copy()

# Keep raw 0–1 values in the saved CSV; display percentages for readability
# where appropriate.
display_df = headline_df.copy()
for col in [
    "accuracy",
    "balanced_accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "weighted_f1",
]:
    display_df[col] = (display_df[col].astype(float) * 100).round(2)

display_df["mcc"] = display_df["mcc"].astype(float).round(4)
display_df["cohen_kappa"] = display_df["cohen_kappa"].astype(float).round(4)

print("Headline comparison (percentage metrics shown as %):")
display(display_df)


In [ ]:
# Cell 08: The three key experimental effects

metric_index = headline_df.set_index("experiment")
KEY_METRICS = ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "mcc"]

def delta_row(name, after, before):
    row = {"comparison": name}
    for metric in KEY_METRICS:
        # Scores live on a 0–1 scale. Multiplying by 100 gives percentage-point
        # change for easier reporting.
        row[f"{metric}_delta_pp"] = (
            float(metric_index.loc[after, metric])
            - float(metric_index.loc[before, metric])
        ) * 100
    return row

delta_df = pd.DataFrame([
    delta_row(
        "Federation effect: FedAvg-IID - Centralized",
        "FedAvg-IID",
        "Centralized",
    ),
    delta_row(
        "Non-IID effect: FedAvg-Non-IID - FedAvg-IID",
        "FedAvg-Non-IID",
        "FedAvg-IID",
    ),
    delta_row(
        "FedProx effect: FedProx-Non-IID - FedAvg-Non-IID",
        "FedProx-Non-IID",
        "FedAvg-Non-IID",
    ),
])

display(delta_df.round(3))


In [ ]:
# Cell 09: Headline metric plot
plot_metrics = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
    "mcc",
]

plot_df = headline_df.set_index("experiment")[plot_metrics].astype(float)

ax = plot_df.plot(kind="bar", figsize=(11, 5))
ax.set_ylabel("Score")
ax.set_xlabel("")
ax.set_ylim(0, 1)
ax.set_title("Centralized vs Federated IDS — Headline Metrics")
ax.legend(title="Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "headline_metrics_comparison.png", dpi=180)
plt.show()


In [ ]:
# Cell 10: Federated convergence comparison
round_files = {
    "FedAvg-IID": paths["iid_rounds"],
    "FedAvg-Non-IID": paths["non_iid_rounds"],
    "FedProx-Non-IID": paths["fedprox_rounds"],
}

round_frames = []
for experiment, path in round_files.items():
    df = pd.read_csv(path).copy()
    df["experiment"] = experiment
    round_frames.append(df)

rounds_df = pd.concat(round_frames, ignore_index=True)

plt.figure(figsize=(8, 5))
for experiment in ["FedAvg-IID", "FedAvg-Non-IID", "FedProx-Non-IID"]:
    part = rounds_df[rounds_df["experiment"] == experiment]
    plt.plot(part["round"], part["macro_f1"], marker="o", label=experiment)

plt.xlabel("Federated round")
plt.ylabel("Global test Macro F1")
plt.title("Federated Convergence — Macro F1")
plt.xticks(sorted(rounds_df["round"].unique()))
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "federated_macro_f1_convergence.png", dpi=180)
plt.show()

plt.figure(figsize=(8, 5))
for experiment in ["FedAvg-IID", "FedAvg-Non-IID", "FedProx-Non-IID"]:
    part = rounds_df[rounds_df["experiment"] == experiment]
    plt.plot(part["round"], part["accuracy"], marker="o", label=experiment)

plt.xlabel("Federated round")
plt.ylabel("Global test accuracy")
plt.title("Federated Convergence — Accuracy")
plt.xticks(sorted(rounds_df["round"].unique()))
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "federated_accuracy_convergence.png", dpi=180)
plt.show()


In [ ]:
# Cell 11: Per-class F1 comparison

per_class_sources = {
    "Centralized": paths["central_per_class"],
    "FedAvg-IID": paths["iid_per_class"],
    "FedAvg-Non-IID": paths["non_iid_per_class"],
    "FedProx-Non-IID": paths["fedprox_per_class"],
}

per_class_frames = []
for experiment, path in per_class_sources.items():
    df = pd.read_csv(path)[["class_id", "class_name", "f1", "support"]].copy()
    df["experiment"] = experiment
    per_class_frames.append(df)

per_class_long = pd.concat(per_class_frames, ignore_index=True)

# Verify the test-set support is identical for each class across experiments.
support_check = (
    per_class_long.groupby("class_name")["support"]
    .nunique()
    .eq(1)
    .all()
)
print("Identical per-class test support across all experiments:", support_check)

per_class_wide = (
    per_class_long
    .pivot(index=["class_id", "class_name"], columns="experiment", values="f1")
    .reset_index()
)

# Explicit column order.
per_class_wide = per_class_wide[
    ["class_id", "class_name"] + EXPERIMENT_ORDER
]

# Direct FedProx-vs-FedAvg non-IID class-level effect.
per_class_wide["FedProx_minus_FedAvg_NonIID"] = (
    per_class_wide["FedProx-Non-IID"]
    - per_class_wide["FedAvg-Non-IID"]
)

display(per_class_wide.sort_values("FedProx_minus_FedAvg_NonIID").head(10))


In [ ]:
# Cell 12: Per-class FedProx effect under the same non-IID partition

effect_df = per_class_wide.sort_values("FedProx_minus_FedAvg_NonIID")

plt.figure(figsize=(10, 9))
plt.barh(
    effect_df["class_name"],
    effect_df["FedProx_minus_FedAvg_NonIID"],
)
plt.axvline(0, linewidth=1)
plt.xlabel("F1 change: FedProx - FedAvg")
plt.ylabel("Class")
plt.title("Per-Class Effect of FedProx Under Non-IID Data")
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fedprox_vs_fedavg_non_iid_per_class_f1.png", dpi=180)
plt.show()


In [ ]:
# Cell 13: Descriptive training-time table
# Runtime is useful context, but only compare it directly if runs used comparable
# Kaggle hardware/runtime conditions.

runtime_rows = [{
    "experiment": "Centralized",
    "training_seconds": central_json["runtime_seconds"]["training"],
}]

for label, data in [
    ("FedAvg-IID", iid_json),
    ("FedAvg-Non-IID", non_iid_json),
    ("FedProx-Non-IID", fedprox_json),
]:
    runtime_rows.append({
        "experiment": label,
        "training_seconds": data["runtime_seconds"]["round_training_total"],
    })

runtime_df = pd.DataFrame(runtime_rows)
runtime_df["training_minutes"] = runtime_df["training_seconds"] / 60

display(runtime_df.round(2))
print(
    "Note: runtime is descriptive. Treat it as a controlled comparison only "
    "when the runs used comparable Kaggle hardware and conditions."
)


In [ ]:
# Cell 14: Save final report-ready artifacts

headline_path = OUTPUT_DIR / "final_headline_metrics.csv"
delta_path = OUTPUT_DIR / "final_experimental_effects.csv"
config_path = OUTPUT_DIR / "final_experiment_configuration.csv"
fairness_path = OUTPUT_DIR / "final_fairness_checks.csv"
rounds_path = OUTPUT_DIR / "final_federated_round_history.csv"
per_class_path = OUTPUT_DIR / "final_per_class_f1_comparison.csv"
runtime_path = OUTPUT_DIR / "final_runtime_summary.csv"
summary_path = OUTPUT_DIR / "final_comparison_summary.json"

headline_df.to_csv(headline_path, index=False)
delta_df.to_csv(delta_path, index=False)
config_df.to_csv(config_path, index=False)
fairness_df.to_csv(fairness_path, index=False)
rounds_df.to_csv(rounds_path, index=False)
per_class_wide.to_csv(per_class_path, index=False)
runtime_df.to_csv(runtime_path, index=False)

summary = {
    "experiments": EXPERIMENT_ORDER,
    "all_control_checks_passed": bool(fairness_df["consistent"].all()),
    "failed_control_checks": failed_checks,
    "headline_metrics": headline_df.to_dict(orient="records"),
    "experimental_effects_percentage_points": delta_df.to_dict(orient="records"),
    "fedavg_non_iid_and_fedprox_exact_partition_match": bool(same_partition_arrays),
    "per_class_support_identical": bool(support_check),
    "runtime_note": (
        "Training runtime is descriptive unless experiments were run under "
        "comparable Kaggle hardware/runtime conditions."
    ),
}

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=str)

print("Saved:")
for path in [
    headline_path,
    delta_path,
    config_path,
    fairness_path,
    rounds_path,
    per_class_path,
    runtime_path,
    summary_path,
]:
    print("-", path)


In [ ]:
# Cell 15: Print concise report facts from the measured results

def pp(after, before, metric):
    return (
        float(metric_index.loc[after, metric])
        - float(metric_index.loc[before, metric])
    ) * 100

print("REPORT-READY FACTS")
print("------------------")
print(
    f"FedAvg-IID vs Centralized: Macro F1 change = "
    f"{pp('FedAvg-IID', 'Centralized', 'macro_f1'):+.2f} pp."
)
print(
    f"FedAvg Non-IID vs IID: Macro F1 change = "
    f"{pp('FedAvg-Non-IID', 'FedAvg-IID', 'macro_f1'):+.2f} pp."
)
print(
    f"FedProx vs FedAvg under the same Non-IID partition: Macro F1 change = "
    f"{pp('FedProx-Non-IID', 'FedAvg-Non-IID', 'macro_f1'):+.2f} pp."
)
print(
    f"Exact FedAvg/FedProx Non-IID partition match: {same_partition_arrays}."
)
print(
    f"All comparison-control checks passed: {fairness_df['consistent'].all()}."
)


In [ ]:
# Cell 16: Package the final comparison outputs

zip_path = Path("/kaggle/working/final_comparison_all_outputs.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in OUTPUT_DIR.iterdir():
        if file.is_file():
            z.write(file, arcname=file.name)

print("Created:", zip_path)
print("Zip size:", f"{zip_path.stat().st_size / 1024**2:.2f} MB")


## Main files for the paper/report

Keep these especially:

- `final_headline_metrics.csv`
- `final_experimental_effects.csv`
- `final_experiment_configuration.csv`
- `final_fairness_checks.csv`
- `final_federated_round_history.csv`
- `final_per_class_f1_comparison.csv`
- `final_comparison_summary.json`
- `final_comparison_all_outputs.zip`

The notebook intentionally separates **measured performance differences** from **fairness/control checks**, so the later report can state both the result and whether the experimental comparison was actually controlled.
